# Evaluation Harness Overview
Based on https://github.com/EleutherAI/lm-evaluation-harness/blob/main/examples/lm-eval-overview.ipynb

Evaluation Harness provides a unified framework for benchmarking LLMs across heterogeneous datasets and task families.  
It enforces a *standardized evaluation pipeline*, allowing different models to be compared rigorously under identical conditions.

At its core, the harness handles the following sequence:

1. **Dataset Loading**  
   Fetches raw samples from HuggingFace datasets, local directories, or custom formats.

2. **Sample Formatting**  
   Converts dataset entries into model-ready prompts via `doc_to_text`.

3. **Target Extraction**  
   Defines ground-truth labels via `doc_to_target`.

4. **Model Interaction (LM Interface)**  
   All models conform to a uniform API:
   - `loglikelihood(context, continuation)`
   - `loglikelihood_rolling(text)`
   - `generate(prompt, max_tokens)`

5. **Metric Computation**  
   Accuracy, exact match, F1, BLEU, ROUGE, or custom scoring modules.

6. **Aggregation & Reporting**  
   Weighted averaging across subtasks, difficulty splits, or domains.

This structure isolates task specification from model behavior, ensuring maximal reproducibility and comparability.


In [ ]:
# Environment for week11: Decoding strategies, evaluation harness, LLM-as-a-judge.
# Colab or a fresh virtualenv: run this cell once. Locally: `uv sync --group week11` in the repo root instead.
%pip install -q "torch>=2.4,<3" "transformers>=4.56,<5" "datasets>=3.6,<4" "accelerate>=1.0" "pandas>=2.2" "tqdm>=4.66" "huggingface-hub>=0.25" "lm-eval>=0.4.8" "ipython>=8"

This notebook provides a structured tutorial covering:

1. How the Harness loads and represents evaluation tasks  
2. How YAML configurations define task behavior  
3. How prompts, targets, and evaluation metrics are constructed  
4. How to modify or create your own tasks  
5. How different evaluation modes (log-likelihood, multiple-choice, generation) operate internally

Each section progressively reveals how the Harness transforms raw datasets into standardized evaluation protocols.


## Creating Evaluation Tasks with Configuration Files

LM-Eval’s most powerful feature is the ability to define tasks entirely through **YAML configuration**, without writing any Python code.

A configuration file specifies:

- dataset path and split  
- how to extract the prompt from each example (`doc_to_text`)  
- how to extract the ground truth (`doc_to_target`)  
- what metric(s) to compute  
- evaluation type (multiple-choice, generative, loglikelihood, etc.)

This decouples task definitions from the evaluation engine and enables clean, reproducible, version-controlled benchmarking.


Here is a minimal YAML example defining a multiple-choice task:

- **`dataset_path` / `dataset_name`** specify the source data  
- **`doc_to_text`** constructs the model input  
- **`doc_to_target`** extracts the correct label  
- **`choices`** enumerates the available options  
- **`metric_list`** specifies accuracy or other metrics  
- **`fewshot_config`** optionally injects few-shot demonstrations

This allows you to describe complex evaluation logic declaratively rather than via procedural Python.


In [ ]:
YAML_boolq_string = """
task: demo_boolq
dataset_path: super_glue
dataset_name: boolq
output_type: multiple_choice
training_split: train
validation_split: validation
doc_to_text: "{{passage}}\nQuestion: {{question}}?\nAnswer:"
doc_to_target: label
doc_to_choice: ["no", "yes"]
should_decontaminate: true
doc_to_decontamination_query: passage
metric_list:
  - metric: acc
"""
with open("boolq.yaml", "w") as f:
    f.write(YAML_boolq_string)

After defining the YAML, evaluation becomes a single command.  
LM-Eval loads the configuration, initializes the dataset, formats prompts, queries the model, and computes metrics automatically.

This pattern supports scalable, repeatable evaluations with versioned task definitions.


In [ ]:
%env LOGLEVEL=DEBUG
!lm_eval \
    --model hf \
    --model_args pretrained="HuggingFaceTB/SmolLM2-360M" \
    --include_path ./ \
    --tasks demo_boolq \
    --limit 10

env: LOGLEVEL=DEBUG
2025-10-20 15:31:39.760159: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760974299.779827     551 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760974299.785615     551 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760974299.800576     551 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760974299.800608     551 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760974299.800612     551 computation_placer.cc:177] co

Tasks can also be grouped into **benchmark suites**, such as:

- MMLU  
- ARC  
- HellaSwag  
- TruthfulQA  

Groups allow you to aggregate metrics across multiple related tasks, producing composite scores or leaderboard metrics.  
This makes it possible to compare models holistically instead of task-by-task.


In [ ]:
YAML_cola_string = """
tag: yes_or_no_tasks
task: demo_cola
dataset_path: glue
dataset_name: cola
output_type: multiple_choice
training_split: train
validation_split: validation
doc_to_text: "{{sentence}}\nQuestion: Does this sentence make sense?\nAnswer:"
doc_to_target: label
doc_to_choice: ["no", "yes"]
should_decontaminate: true
doc_to_decontamination_query: sentence
metric_list:
  - metric: acc
"""
with open("cola.yaml", "w") as f:
    f.write(YAML_cola_string)

In [ ]:
# !accelerate launch --no_python
%env LOGLEVEL=DEBUG
!lm_eval \
    --model hf \
    --model_args pretrained="HuggingFaceTB/SmolLM2-360M" \
    --include_path ./ \
    --tasks yes_or_no_tasks \
    --limit 10 \
    --output output/yes_or_no_tasks/ \
    --log_samples

env: LOGLEVEL=DEBUG
2025-10-20 15:32:26.325760: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760974346.345067     827 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760974346.350918     827 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760974346.365475     827 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760974346.365500     827 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760974346.365504     827 computation_placer.cc:177] co

## Editing Prompt Templates

Templates define how input examples are converted into natural-language prompts.  
Most Harness templates use Jinja2 for:

- injecting fields into text  
- formatting multiple-choice options  
- creating few-shot examples  
- modifying structure without touching code

Because templates are externalized, you can iterate rapidly on prompt engineering without rebuilding the task logic.

In [ ]:
YAML_mmlu_geo_string = """
task: demo_mmlu_high_school_geography
dataset_path: cais/mmlu
dataset_name: high_school_geography
description: "The following are multiple choice questions (with answers) about high school geography.\n\n"
test_split: test
fewshot_split: dev
fewshot_config:
  sampler: first_n
output_type: multiple_choice
doc_to_text: "{{question.strip()}}\nA. {{choices[0]}}\nB. {{choices[1]}}\nC. {{choices[2]}}\nD. {{choices[3]}}\nAnswer:"
doc_to_choice: ["A", "B", "C", "D"]
doc_to_target: answer
metric_list:
  - metric: acc
    aggregation: mean
    higher_is_better: true
  - metric: acc_norm
    aggregation: mean
    higher_is_better: true
"""
with open("mmlu_high_school_geography.yaml", "w") as f:
    f.write(YAML_mmlu_geo_string)

In [ ]:
# !accelerate launch --no_python
%env LOGLEVEL=DEBUG
!lm_eval \
    --model hf \
    --model_args pretrained="HuggingFaceTB/SmolLM2-360M" \
    --include_path ./ \
    --tasks demo_mmlu_high_school_geography \
    --limit 10 \
    --output output/mmlu_high_school_geography/ \
    --log_samples

env: LOGLEVEL=DEBUG
2025-10-20 14:45:14.227649: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760971514.247668    3747 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760971514.253661    3747 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760971514.268733    3747 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760971514.268757    3747 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760971514.268761    3747 computation_placer.cc:177] co

A single dataset may support multiple evaluation *modes*.  
For example:

- **MC1 / MC2** evaluate the log-likelihood of the correct option  
- **Generative mode** evaluates the model’s ability to produce the continuation directly  
- **Rank-based scoring** evaluates preference across options

This flexibility allows a single dataset to test multiple capabilities of the model.

In [ ]:
YAML_mmlu_geo_string = """
include: mmlu_high_school_geography.yaml
task: demo_mmlu_high_school_geography_continuation
doc_to_text: "{{question.strip()}}\nA. {{choices[0]}}\nB. {{choices[1]}}\nC. {{choices[2]}}\nD. {{choices[3]}}\nAnswer:"
doc_to_choice: "{{choices}}"
"""
with open("mmlu_high_school_geography_continuation.yaml", "w") as f:
    f.write(YAML_mmlu_geo_string)

In [ ]:
# !accelerate launch --no_python
%env LOGLEVEL=DEBUG
!lm_eval \
    --model hf \
    --model_args pretrained="HuggingFaceTB/SmolLM2-360M" \
    --include_path ./ \
    --tasks demo_mmlu_high_school_geography_continuation \
    --limit 10 \
    --output output/mmlu_high_school_geography_continuation/ \
    --log_samples

env: LOGLEVEL=DEBUG
2025-10-20 14:42:23.951704: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760971343.970920    2948 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760971343.976983    2948 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760971343.991704    2948 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760971343.991729    2948 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760971343.991733    2948 computation_placer.cc:177] co

Inspecting samples helps verify that `doc_to_text`, `doc_to_target`, and `choices` behave as expected.  
Correct formatting is crucial, since even small inconsistencies in prompts or labels can materially impact evaluation scores.

By examining examples, you ensure that the constructed prompt matches the intended evaluation design.


## Closer Look at YAML Fields

A full LM-Eval task definition consists of several key fields:

- **Metadata** — task name, description, reference links  
- **Dataset** — where and how to load examples  
- **Document transformation** — `doc_to_text`, `doc_to_target`, optional pre-processing  
- **Few-shot parameters** — number of examples, formatting rules  
- **Evaluation type** — multiple-choice, generative, loglikelihood, etc.  
- **Metrics** — accuracy, exact match, BLEU, ROUGE, F1, log-likelihood averages  
- **Filters** — removing empty samples, skipping malformed data  

Understanding these fields makes it straightforward to author your own tasks.


## When Jinja Templates Are Not Enough

Some tasks require dynamic logic that cannot be written cleanly in a Jinja template.  
In these cases, LM-Eval supports:

1. **Custom Python functions** via `!function`  
2. **Dataset preprocessing hooks**  
3. **Arbitrary programmatic transformations**

These mechanisms allow you to encode more sophisticated logic while still benefiting from LM-Eval’s standardized evaluation flow.


Below is an illustration of how `!function` can be used to construct a custom `doc_to_text` or `doc_to_target`.  
This enables richer prompt structures, parsing, conditional logic, or multi-field assembly directly inside the pre-processing pipeline.


In [ ]:
YAML_mmlu_geo_string = """
include: mmlu_high_school_geography.yaml
task: demo_mmlu_high_school_geography_function_prompt
doc_to_text: !function utils.doc_to_text
doc_to_choice: "{{choices}}"
"""
with open("demo_mmlu_high_school_geography_function_prompt.yaml", "w") as f:
    f.write(YAML_mmlu_geo_string)

DOC_TO_TEXT = """
def doc_to_text(x):
    question = x["question"].strip()
    choices = x["choices"]
    option_a = choices[0]
    option_b = choices[1]
    option_c = choices[2]
    option_d = choices[3]
    return f"{question}\\nA. {option_a}\\nB. {option_b}\\nC. {option_c}\\nD. {option_d}\\nAnswer:"
"""
with open("utils.py", "w") as f:
    f.write(DOC_TO_TEXT)

!lm_eval \
    --model hf \
    --model_args pretrained="HuggingFaceTB/SmolLM2-360M" \
    --include_path ./ \
    --tasks demo_mmlu_high_school_geography_function_prompt \
    --limit 10 \
    --output output/demo_mmlu_high_school_geography_function_prompt/ \
    --log_samples

2025-10-20 14:47:44.319091: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760971664.338874    4451 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760971664.345556    4451 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760971664.363008    4451 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760971664.363033    4451 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760971664.363039    4451 computation_placer.cc:177] computation placer alr

Preprocessing can also be applied directly to the dataset before evaluation.  
This allows you to:

- add derived fields  
- normalize text  
- split multi-part inputs  
- restructure complex samples into simpler evaluation-ready forms

The preprocessed dataset can then be saved and reused across tasks.


In [ ]:
YAML_mmlu_geo_string = """
include: mmlu_high_school_geography.yaml
task: demo_mmlu_high_school_geography_function_prompt_2
process_docs: !function utils_process_docs.process_docs
doc_to_text: "{{input}}"
doc_to_choice: "{{choices}}"
"""
with open("demo_mmlu_high_school_geography_process_docs.yaml", "w") as f:
    f.write(YAML_mmlu_geo_string)

DOC_TO_TEXT = """
def process_docs(dataset):
    def _process_doc(x):
        question = x["question"].strip()
        choices = x["choices"]
        option_a = choices[0]
        option_b = choices[1]
        option_c = choices[2]
        option_d = choices[3]
        out_doc = {}
        out_doc["input"] = f"{question}\\nA. {option_a}\\nB. {option_b}\\nC. {option_c}\\nD. {option_d}\\nAnswer:"
        return out_doc

    return dataset.map(_process_doc)
"""

with open("utils_process_docs.py", "w") as f:
    f.write(DOC_TO_TEXT)

!lm_eval \
    --model hf \
    --model_args pretrained="HuggingFaceTB/SmolLM2-360M" \
    --include_path ./ \
    --tasks demo_mmlu_high_school_geography_function_prompt_2 \
    --limit 10 \
    --output output/demo_mmlu_high_school_geography_function_prompt_2/ \
    --log_samples

2025-10-20 14:49:28.372685: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760971768.392543    4980 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760971768.398469    4980 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760971768.413809    4980 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760971768.413835    4980 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760971768.413839    4980 computation_placer.cc:177] computation placer alr

This tutorial covered the essentials of configuring tasks, manipulating templates, and preparing datasets.  
For more detailed documentation, advanced examples, and integration guides, consult the LM-Eval docs in the repository’s [docs/](https://github.com/EleutherAI/lm-evaluation-harness/tree/main/docs) directory.

# How to Add Your Custom Task

To create a new task:

1. Prepare a dataset (local or Hugging Face Hub).  
2. Write a YAML configuration specifying formatting and metrics.  
3. (Optional) Add preprocessing or custom functions.  
4. Run LM-Eval with your configuration.  
5. Validate the output and iterate.

With these tools, you can build robust, reproducible evaluations tailored to your domain.


## Example: Intent Classification Pipeline in LM-Eval

To illustrate how the Evaluation Harness handles *non-generative* tasks, let's walk through the simple intent-classification example defined earlier.

This pipeline has four conceptual stages:

---

### 1. **Dataset Construction**

We first assemble a small dictionary-based dataset:

```python
DUMMY_INTENT_DATA = {
    "train": {...},
    "test": {...},
}
```

and convert it to a Hugging Face `DatasetDict`:

```python
intent_cls = datasets.DatasetDict({
    "train": datasets.Dataset.from_dict(DUMMY_INTENT_DATA["train"]),
    "test": datasets.Dataset.from_dict(DUMMY_INTENT_DATA["test"]),
})
```

At this point, the data is still *raw*. LM-Eval knows nothing about prompts, labels, or metrics.

---

### 2. **Task Definition via YAML**

The YAML configuration specifies how examples are transformed during evaluation:

- **`doc_to_text`** builds the input prompt string  
- **`doc_to_target`** extracts the correct intent label  
- **`choices`** enumerates the allowed classes  
- **`metric_list`** specifies accuracy as the scoring metric  
- **`task`** and **`dataset_path`** tell LM-Eval where this task lives

This is where the “model sees this as a classification problem” is made explicit.

Example logic inside `doc_to_text`:

```
Intent: {{text}}
Options: {{choices}}
Classify the intent:
```

Example logic inside `doc_to_target`:

```
{{label}}
```

This ensures the model’s interface is *exactly identical* across architectures.

---

### 3. **Model Scoring**

During evaluation, LM-Eval does **not** ask the model to generate an entire sentence.  
Instead, for classification tasks:

1. It computes the **log-likelihood** of each candidate label (string form).  
2. Compares those likelihoods.  
3. Selects the label with the highest score.

Effectively:

$$
\text{predicted-class} = \arg\max_{c \in \text{choices}} \log p(c \mid \text{prompt})
$$

This makes the evaluation deterministic and architecture-agnostic.

---

### 4. **Metrics and Aggregation**

Because it's a classification task, the metric is straightforward:

- **Accuracy** = proportion of correct predictions  
- Optionally macro/micro averages if there are multiple intent types

LM-Eval aggregates results over the test split and outputs a standardized JSON summary.

---

### Why This Matters

This pipeline makes classification tasks:

- **Repeatable** — prompt, labels, and scoring are frozen in the config.  
- **Comparable** — both small and giant models are evaluated identically.  
- **Debuggable** — you can inspect every intermediate (prompt, targets, per-choice scores).  
- **Extensible** — adding new labels or new intent datasets requires only a YAML edit.

This is the minimal example that demonstrates the Harness’s real value: **the logic of evaluation is declarative, not embedded in code**, which is exactly how large benchmark suites remain sane over time.


In [ ]:
import datasets

In [ ]:
# Simple intent classification benchmark
DUMMY_INTENT_DATA = {
    "train": {
        "sentence": ["I need a flight to Paris tomorrow morning",
                     "What's the weather in Berlin?",
                     "Set a reminder for 5 pm to call mom",
                     "Play some jazz music",
                     "Find me a good Italian restaurant nearby",],
        "label": ["book_flight", "weather_query", "set_reminder", "play_music", "find_restaurant"],
    },
    "test": {
        "sentence": ["Book me a ticket to New York tonight",
                     "Is it going to rain tomorrow?",
                     "Remind me at 9 to send the report",
                     "Put on some classical tunes",
                     "Any sushi places around here?",],
        "label": ["book_flight", "weather_query", "set_reminder", "play_music", "find_restaurant"],
    },
}

INTENT_LABELS = [
    "book_flight",
    "weather_query",
    "set_reminder",
    "play_music",
    "find_restaurant",
]
int_cls = datasets.DatasetDict({"train": datasets.Dataset.from_dict(DUMMY_INTENT_DATA["train"]),
                                "test": datasets.Dataset.from_dict(DUMMY_INTENT_DATA["test"])})

In [ ]:
int_cls.save_to_disk("./intent_classification")

Saving the dataset (0/1 shards):   0%|          | 0/5 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
int_cls

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 5
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 5
    })
})

In [ ]:
int_cls = datasets.load_from_disk("./intent_classification")
int_cls

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 5
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 5
    })
})

In [ ]:
del int_cls

In [ ]:
YAML_cola_string = """
task: int_cls
dataset_path: arrow
dataset_kwargs:
  data_files:
    train: ./intent_classification/train/data-00000-of-00001.arrow
    test: ./intent_classification/test/data-00000-of-00001.arrow
output_type: multiple_choice
training_split: train
validation_split: test
doc_to_text: "{{sentence}}\nQuestion: Define the intent of the query on five possible types: book_flight, weather_query, set_reminder, play_music, find_restaurant.\nAnswer:"
doc_to_target: label
doc_to_choice: ["book_flight", "weather_query", "set_reminder", "play_music", "find_restaurant"]
should_decontaminate: true
doc_to_decontamination_query: sentence
metric_list:
  - metric: acc
"""
with open("int_cls.yaml", "w") as f:
    f.write(YAML_cola_string)


In [ ]:
# !accelerate launch --no_python
%env LOGLEVEL=DEBUG
!lm_eval \
    --model hf \
    --model_args pretrained="HuggingFaceTB/SmolLM2-360M" \
    --include_path ./ \
    --tasks int_cls \
    --limit 5 \
    --output output/int_cls/ \
    --log_samples

env: LOGLEVEL=DEBUG
2025-10-20 15:35:53.845850: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760974553.865301    1866 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760974553.871185    1866 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760974553.885918    1866 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760974553.885946    1866 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760974553.885950    1866 computation_placer.cc:177] co

In [ ]:
YAML_cola_string = """
task: int_cls_gen
dataset_path: arrow
dataset_kwargs:
  data_files:
    train: ./intent_classification/train/data-00000-of-00001.arrow
    test: ./intent_classification/test/data-00000-of-00001.arrow
output_type: generate_until
training_split: train
validation_split: test
doc_to_text: "{{sentence}}\nQuestion: Define the intent of the query on five possible types: book_flight, weather_query, set_reminder, play_music, find_restaurant.\nAnswer:"
doc_to_target: "{{label}}"
metric_list:
  - metric: exact_match
    higher_is_better: true
    ignore_case: true
    ignore_punctuation: false
    regexes_to_ignore:
      - ","
      - "$"
generation_kwargs:
  until:
    - "Question:"
    - "</s>"
    - "<|im_end|>"
  do_sample: false
  temperature: 0.0"""
with open("int_cls_gen.yaml", "w") as f:
    f.write(YAML_cola_string)


In [ ]:
# !accelerate launch --no_python
%env LOGLEVEL=DEBUG
!lm_eval \
    --model hf \
    --model_args pretrained="HuggingFaceTB/SmolLM2-360M" \
    --include_path ./ \
    --tasks int_cls_gen \
    --limit 5 \
    --output output/int_cls_gen/ \
    --log_samples

env: LOGLEVEL=DEBUG
2025-10-20 15:50:36.104505: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760975436.124501    5698 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760975436.134183    5698 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760975436.156631    5698 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760975436.156658    5698 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760975436.156662    5698 computation_placer.cc:177] co